### __Tools__

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5


In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number."""
    return x ** 0.5


In [5]:
@tool("square_root", description="Calculate the square root of a number.")
def tool1(x: float) -> float:
    return x ** 0.5


In [6]:
tool1.invoke({"x": 467})

21.61018278497431

#### __Adding to agents__

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You're an arithmetic wizard. Use your tools to calculate the square root of any number I give you."
)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What's the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response["messages"][-1].content)

The square root of 467 is approximately 21.61018278497431. 
(≈ 21.6102 when rounded to four decimal places)


In [9]:
from pprint import pprint

pprint(response["messages"])

[HumanMessage(content="What's the square root of 467?", additional_kwargs={}, response_metadata={}, id='e52d9e62-b26c-4d04-8337-523f59801617'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 536, 'prompt_tokens': 157, 'total_tokens': 693, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DhcoTTmwBD3ExuymbyloMZwHSdzGV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e45ee-ef59-7e91-b662-0fd2cb06ad20-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_lJvqE2sxho2bVFU6mfAtcOkq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 536, 'total_toke

In [11]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_lJvqE2sxho2bVFU6mfAtcOkq', 'type': 'tool_call'}]


#### __Add web search tool__

In [12]:
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for the given query and return the results."""
    return tavily_client.search(query)

web_search.invoke({"query": "What is the current mayor of Campina Grande, Paraiba?"})

{'query': 'What is the current mayor of Campina Grande, Paraiba?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.db-city.com/Brazil--Para%C3%ADba--Campina-Grande',
   'title': 'Campina Grande, Paraíba, Brazil - City, Town and Village of the world',
   'content': 'Campina Grande ; Administration · Campina Grande Mayor, ROMERO RODRIGUES VEIGA ; Demography. Information on the people and the population of Campina Grande.',
   'score': 0.9997713,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/2026_Para%C3%ADba_general_election',
   'title': '2026 Paraíba general election - Wikipedia',
   'content': 'Lucas Ribeiro, lawyer, current vice-governor of Paraíba (since 2023), former vice-mayor of Campina Grande (2021–2022), former City Councilor of Campina Grande (',
   'score': 0.9979007,
   'raw_content': None},
  {'url': 'https://www.instagram.com/p/DYCjYtHFvFP/',
   'title': 'O vereador de Campina Grande/PB, Márcio da Eletropol

In [14]:
web_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
)

question = HumanMessage(content="Who is the current mayor of Campina Grande, Paraiba?")

response = web_agent.invoke(
    {"messages": [question]}
)

In [15]:
pprint(response["messages"])

[HumanMessage(content='Who is the current mayor of Campina Grande, Paraiba?', additional_kwargs={}, response_metadata={}, id='d5f441e2-882b-49b3-9d9d-8aefbfbc8e54'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 143, 'total_tokens': 301, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DhfFTWQlEvhuGCGssvXEgpZaN1pp3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e467d-c23c-7b62-b73e-7842f307fe00-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of Campina Grande Paraíba'}, 'id': 'call_TIedv7iMu9nwyRNh1z9GLE8p', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [17]:
response["messages"][-1].content

'Bruno Cunha Lima, of União Brasil. He has served as the mayor since taking office in January 2021 and is the current mayor as of 2026.\n\nSources:\n- Prefeitura de Campina Grande – O Prefeito: https://campinagrande.pb.gov.br/o-prefeito/\n- Lista de prefeitos de Campina Grande – Wikipedia: https://pt.wikipedia.org/wiki/Lista_de_prefeitos_de_Campina_Grande'